# GPU Programming with CUDA — A Primer for CPU Programmers

Most of the floating-point performance of today's supercomputers comes from GPUs.
A GPU is a parallel processor, too — but one that was designed with a different goal than a CPU, and it shows in every detail: in the hardware, in the programming model, and above all in the way it deals with the latency of memory.

This notebook introduces CUDA, NVIDIA's programming model for its GPUs, from the perspective of someone who knows CPUs.
Throughout, we put the two side by side: what is the same, what only *sounds* the same, and what is fundamentally different.

**After working through this notebook you will be able to**

* explain why CPUs and GPUs look so different, starting from latency vs. throughput,
* translate the GPU vocabulary — SM, warp, block, grid, "CUDA core", shared and local memory — into CPU terms, and spot where the analogy breaks,
* write, launch and time simple CUDA kernels and move data between host and device,
* explain how a GPU schedules work — blocks to SMs, warps to issue slots — and how this differs from out-of-order execution and OS scheduling on a CPU,
* explain latency hiding by many threads (GPU) vs. instruction-level parallelism (CPU), estimate the required parallelism with Little's law, and reason about occupancy,
* recognize the GPU's performance pitfalls — divergence and uncoalesced memory access — and their CPU counterparts,
* use streams for asynchronous execution, and see how all this connects to MPI.

As before, prompts marked with 🤔 **Predict** ask you to think about the result *before* running the next cell.

## Working with this Notebook

The CUDA cells are compiled with `nvcc` and run by the `%%cuda` cell magic, which works like `%%c` from MPI in a Nutshell.
They need an **NVIDIA GPU and the CUDA toolkit** — on the cluster, start JupyterLab on a GPU node and load the CUDA module.
Without `nvcc`, the CUDA cells are skipped with a message; read their code and the discussion anyway.
The CPU cells (`%%c`) and the Python cells run everywhere, so you can do all the CPU experiments on your laptop.

| Option | Meaning |
|---|---|
| `-r` | the cell is a complete program — needed for every cell with a kernel, since kernels are defined outside of `main()` |
| `-i cuda-check.h -f Icode` | include the helper header written below: error checks and a GPU timer |
| `-o code/….cu` | where to save the source; the extension `.cu` tells `nvcc` that it is CUDA |
| `-F O3 arch=sm_90` | replace the default flags `-O3 -arch=native -std=c++17`, e.g. to compile for a GPU that is not in this machine |
| `-f Xptxas=-v` | additional flags, here: report the registers used per thread |

CUDA code is **C++**: C code compiles, too, but e.g. the result of `malloc()` must be cast.

Load the extension once per kernel session:

In [ ]:
%load_ext ice.magic

Every CUDA runtime call returns an error code, and errors are easily missed — e.g. if a kernel did not run at all.
The following helper header checks every call with `CUDA_CHECK()`, and provides a timer based on CUDA *events* (explained with the first measurement).
Run the cell to create the header:

In [ ]:
%%writefile code/cuda-check.h
// Helpers for the CUDA examples of the PAMPI notebooks.
#pragma once
#include <stdio.h>
#include <stdlib.h>

// Every CUDA runtime call returns an error code - check it. Kernel launches return nothing,
// their errors (e.g. too many threads per block) are fetched with cudaGetLastError().
#define CUDA_CHECK(call)                                                            \
    do {                                                                            \
        cudaError_t err_ = (call);                                                  \
        if (err_ != cudaSuccess) {                                                  \
            fprintf(stderr, "%s:%d: %s failed: %s\n", __FILE__, __LINE__, #call,    \
                    cudaGetErrorString(err_));                                      \
            exit(1);                                                                \
        }                                                                           \
    } while (0)
#define CUDA_CHECK_LAUNCH() CUDA_CHECK(cudaGetLastError())

// Times GPU work with two events, which the GPU timestamps when it reaches them in the stream:
//     GpuTimer timer;
//     timer.start();
//     kernel<<<blocks, threads>>>(...);
//     float ms = timer.stop();     // waits until the kernel has finished
struct GpuTimer {
    cudaEvent_t begin, end;
    GpuTimer()  { CUDA_CHECK(cudaEventCreate(&begin)); CUDA_CHECK(cudaEventCreate(&end)); }
    ~GpuTimer() { cudaEventDestroy(begin); cudaEventDestroy(end); }
    void start(cudaStream_t stream = 0) { CUDA_CHECK(cudaEventRecord(begin, stream)); }
    float stop(cudaStream_t stream = 0) {
        CUDA_CHECK(cudaEventRecord(end, stream));
        CUDA_CHECK(cudaEventSynchronize(end));
        float ms;
        CUDA_CHECK(cudaEventElapsedTime(&ms, begin, end));
        return ms;
    }
};

// the number of SMs of the current GPU
static inline int sm_count(void) {
    int device, count;
    CUDA_CHECK(cudaGetDevice(&device));
    CUDA_CHECK(cudaDeviceGetAttribute(&count, cudaDevAttrMultiProcessorCount, device));
    return count;
}

## Two Design Philosophies

A CPU core is built to execute **one** instruction stream as fast as possible.
The enemy is latency: a load that misses the caches takes about 100 ns — several hundred cycles in which the core could have done useful work.
So the CPU spends most of its transistors on **avoiding** latency (large caches, prefetchers) and **hiding** it within a single thread (branch prediction, speculative and out-of-order execution).

A GPU is built for **throughput**: the total amount of work per second, for problems with millions of independent pieces of work.
A single thread may be slow — what counts is that the hardware is busy.
So the GPU spends its transistors on **arithmetic units and registers**, and hides latency in the simplest possible way: it keeps thousands of threads resident and switches to another one whenever a thread has to wait.

![CPU and GPU chips compared with the same number of units: a CPU core spends its area on control logic and caches for two threads, a streaming multiprocessor on arithmetic lanes and a large register file for 2048 threads; the CPU has a large L3 cache, the GPU a smaller L2 cache but much faster memory](img/gpu-cpu-chip.svg)

Here are the two current server processors of AMD and NVIDIA (vendor specifications, rounded):

| | CPU: AMD EPYC 9965 (Zen 5c) | GPU: NVIDIA B200 (Blackwell) | GPU / CPU |
|---|---|---|---|
| manufacturing process | TSMC 3 nm (N3E) compute dies + 6 nm I/O die | TSMC 4 nm class (4NP), 2 dies in one package | |
| cores / SMs | 192 cores | 148 SMs | 0.8 |
| FP64 lanes (FMA per cycle) | 3072 (16 per core: 2 × AVX-512) | 9472 (64 per SM) | 3.1 |
| FP32 lanes | 6144 | 18 944 (128 per SM) | 3.1 |
| clock | 2.25 GHz (up to 3.7) | up to ≈ 1.96 GHz | 0.9 |
| peak FP64 | 13.8 TFLOP/s | 37 TFLOP/s (40 with tensor cores) | 2.7 |
| peak FP32 | 27.6 TFLOP/s | 75 TFLOP/s | 2.7 |
| memory bandwidth | 0.58 TB/s (12 × DDR5-6000) | 8 TB/s (HBM3e) | 14 |
| memory capacity | up to 6 TB | 180 GB | 1/33 |
| last-level cache | 384 MB L3 | 126 MB L2 | 1/3 |
| registers | ≈ 4.5 MiB (≈ 24 KiB per core) | 37 MiB (256 KiB per SM) | 8 |
| hardware threads | 384 (2 per core) | 303 104 (2048 per SM) | 790 |
| power | 500 W | 1000 W | 2 |
| machine balance (FP64) | 24 flop/byte | 4.6 flop/byte | 1/5 |

The peak rates count the SIMD lanes and the GPU's FP32/FP64 units.
The tensor cores reach far higher rates in the low-precision formats of AI (TF32, BF16, FP8, FP4) — but only for matrix products, so these rates are not comparable.

Some observations:

* **The number of cores and SMs has converged** — the CPU now even has more cores than the GPU has SMs. The difference is *inside* the units: an SM has 4× the FP64 and FP32 lanes of a core, and keeps 1000× as many threads.
  Counting "cores" says little; count lanes and threads.
* The GPU is only about 3× faster in peak FP64: AVX-512 at full width has made the CPU cores much wider, while current GPUs invest their transistors in low-precision tensor cores.
  The large gap is the **memory bandwidth**: 14×.
* The **machine balance** has diverged: the CPU needs about 24 flops per byte loaded to be limited by computation rather than memory, the GPU only about 5.
  Memory-bound codes such as stencils gain about the bandwidth ratio on the GPU, compute-bound FP64 codes far less. Speedups of 100× are usually comparisons with poorly optimized CPU code.
* The CPU is even made in the newer process — the differences are design decisions, not technology.
  And the most striking numbers are in the last rows: 8× the registers, and **almost 800× the threads**. That is where the GPU's design differs — and why.

### How Much Parallelism Is Needed? Little's Law

**Little's law** from queueing theory relates throughput, latency and concurrency:

$$\text{data in flight} = \text{bandwidth} \times \text{latency}$$

To use the full memory bandwidth, enough requests must be on their way *at any time* to cover the latency.
The following cell evaluates this for both machines; the GPU's memory latency is an assumption here — you will measure it below.

In [ ]:
# vendor numbers (rounded) - change them to the values of your machines
machines = {
    'CPU (EPYC 9965)': dict(flops=13.8e12, bandwidth=0.58e12, latency=100e-9, units=192, threads=384),
    'GPU (B200)':      dict(flops=37e12,   bandwidth=8e12,    latency=500e-9, units=148, threads=148 * 2048),
    # your GPU: the device query below prints this line for it, here for an A100
    # 'my GPU':        dict(bandwidth=1555e9, latency=500e-9, units=108, threads=221184),
}

for name, m in machines.items():
    in_flight = m['bandwidth'] * m['latency']       # Little's law: bytes on their way to saturate the bandwidth
    if 'flops' in m:                                # flops per byte at which the code becomes compute-bound
        print(f"{name}: machine balance {m['flops'] / m['bandwidth']:4.1f} flop/byte")
    else:
        print(f"{name}:")
    print(f"    bytes in flight: {in_flight / 1e3:6.0f} kB = {in_flight / m['units'] / 1e3:5.1f} kB per core/SM"
          f" = {in_flight / m['units'] / 64:5.0f} cache lines of 64 bytes per core/SM")
    print(f"    with all hardware threads busy: {in_flight / m['threads']:6.1f} bytes in flight per thread")

A CPU core needs a few cache lines in flight — out-of-order execution and the hardware prefetchers manage that within a single thread.
An SM needs several hundred — and its threads execute in order: a thread that waits for a load can't run ahead to issue hundreds more.
So the SM needs many threads, and even with all 2048 resident, **every thread must have more than a `double` on the way at all times**.
Keep this number in mind; it explains most of what follows.

## A Rosetta Stone: CPU and GPU Nomenclature

GPU vocabulary reuses CPU words with different meanings, and marketing adds to the confusion.
Let's look inside one CPU core and one SM:

![A CPU core compared with a streaming multiprocessor: the core has a large out-of-order engine for one or two hardware threads; the SM has four sub-partitions, each with a simple warp scheduler, a 64 KB register file and 32 lanes, and keeps up to 64 warps resident](img/gpu-core-vs-sm.svg)

### Hardware

| CPU | NVIDIA GPU | Remarks |
|---|---|---|
| processor, socket | GPU, the **device** | the CPU is the **host** |
| core | **streaming multiprocessor (SM)** | the unit with its own instruction scheduling — the true counterpart of a core |
| SIMD lane | "CUDA core" | marketing: a "CUDA core" is one FP32 lane, not a core — the B200's "18 944 CUDA cores" are 148 SMs × 128 lanes |
| SIMD instruction on a vector register (AVX-512: 8 doubles) | one instruction of a **warp**: 32 threads | the GPU's vector width is 32 |
| hardware thread (SMT: 1–2 per core) | **resident warp** (up to 64 per SM) | a hardware context: registers and program counter |
| out-of-order engine | warp scheduler (4 per SM) | in-order issue, no speculation — but a choice of up to 16 warps per cycle |
| register file (≈ 24 KiB per core) | register file (256 KiB per SM) | partitioned among all resident threads |
| L1 cache | L1 cache + **shared memory** (256 KiB per SM in total) | shared memory: a scratchpad managed by the program, not a cache |
| L2, L3 cache | L2 cache, shared by all SMs | no per-SM L2 |
| main memory (DDR) | **global memory** (HBM or GDDR), "device memory" | separate from the host memory |
| — | **local memory** | per-thread memory for register spills and local arrays — in DRAM, despite the name |
| — | PCIe or NVLink to the host | PCIe 5.0 x16: ≈ 64 GB/s, over 100× slower than HBM |

### Execution Model

| CPU (OpenMP, MPI) | CUDA | Remarks |
|---|---|---|
| program | **host code** | runs on the CPU, manages memory, launches kernels |
| body of a parallel loop | **kernel**: a `__global__` function | executed by every thread |
| `#pragma omp parallel for` | kernel launch `kernel<<<blocks, threads>>>(…)` | asynchronous: returns immediately |
| loop iteration `i` | thread; `i = blockIdx.x * blockDim.x + threadIdx.x` | |
| OS thread (MB of stack, µs to create) | CUDA thread: one lane of a warp | created and scheduled by hardware |
| team of threads | **thread block** (≤ 1024 threads) | runs on one SM; barrier `__syncthreads()`; shared memory |
| all iterations of the loop | **grid** of blocks | blocks are independent — no barrier between the blocks of a kernel |
| `omp_get_thread_num()` | `threadIdx`, `blockIdx`, `blockDim`, `gridDim` | up to three dimensions: `.x`, `.y`, `.z` |
| private variable | local variable → register | |
| shared variable | `__shared__` (per block), global memory (all threads) | |
| `#pragma omp barrier` | `__syncthreads()` | only within a block — the end of a kernel is the global barrier |
| `#pragma omp atomic` | `atomicAdd()`, `atomicMax()`, … | |
| nonblocking MPI: `MPI_Isend()`, `MPI_Wait()` | **streams**: asynchronous launches and copies, `cudaStreamSynchronize()` | see below |

### Other Vendors

AMD (HIP/ROCm) and the portable models OpenCL and SYCL use the same concepts under yet other names:

| NVIDIA CUDA | AMD HIP | OpenCL, SYCL |
|---|---|---|
| SM | compute unit (CU) | compute unit |
| warp (32 threads) | wavefront (64 threads on Instinct GPUs) | sub-group |
| thread | thread, work-item | work-item |
| block | block, workgroup | work-group |
| grid | grid | NDRange / `nd_range` |
| shared memory | LDS (local data share) | **local** memory |
| local memory | scratch | private memory |

> ⚠️ **Watch out:** OpenCL's *local* memory is CUDA's *shared* memory — fast and on-chip; CUDA's *local* memory is slow DRAM. Same word, opposite meaning.

Let's see what your GPU looks like in these terms.
The device query asks the CUDA runtime for the properties of the GPU:

In [ ]:
%%cuda -r -i cuda-check.h -f Icode -o code/cuda-device-query.cu
// one property of the GPU, queried by its attribute
static int attribute(cudaDeviceAttr attr, int device) {
    int value;
    CUDA_CHECK(cudaDeviceGetAttribute(&value, attr, device));
    return value;
}

int main(void) {
    int count, dev = 0;
    CUDA_CHECK(cudaGetDeviceCount(&count));
    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, dev));
    printf("%d GPU(s), device %d: %s, compute capability %d.%d\n\n", count, dev, prop.name,
           attribute(cudaDevAttrComputeCapabilityMajor, dev), attribute(cudaDevAttrComputeCapabilityMinor, dev));

    const int sms       = attribute(cudaDevAttrMultiProcessorCount, dev);
    const int warp      = attribute(cudaDevAttrWarpSize, dev);
    const int threadsSM = attribute(cudaDevAttrMaxThreadsPerMultiProcessor, dev);
    const int blocksSM  = attribute(cudaDevAttrMaxBlocksPerMultiprocessor, dev);
    const int regsSM    = attribute(cudaDevAttrMaxRegistersPerMultiprocessor, dev);
    const int smemSM    = attribute(cudaDevAttrMaxSharedMemoryPerMultiprocessor, dev);
    const int l2        = attribute(cudaDevAttrL2CacheSize, dev);
    const int clock     = attribute(cudaDevAttrClockRate, dev);              // kHz
    const int memClock  = attribute(cudaDevAttrMemoryClockRate, dev);        // kHz
    const int busWidth  = attribute(cudaDevAttrGlobalMemoryBusWidth, dev);   // bits

    printf("%-34s %s\n", "GPU term", "value  (CPU counterpart)");
    printf("%-34s %d  (cores)\n", "streaming multiprocessors (SMs)", sms);
    printf("%-34s %d threads  (SIMD width)\n", "warp size", warp);
    printf("%-34s %d threads = %d warps  (hardware threads per core: 1-2)\n", "resident threads per SM", threadsSM, threadsSM / warp);
    printf("%-34s %d\n", "resident blocks per SM", blocksSM);
    printf("%-34s %d\n", "threads per block (max.)", attribute(cudaDevAttrMaxThreadsPerBlock, dev));
    printf("%-34s %d x 32 bit = %d KiB  (a few KiB per core)\n", "registers per SM", regsSM, regsSM * 4 / 1024);
    printf("%-34s %d KiB  (no counterpart; L1 cache: 32-64 KiB)\n", "shared memory per SM (max.)", smemSM / 1024);
    printf("%-34s %.1f MiB  (L2 + L3 cache)\n", "L2 cache", l2 / 1048576.0);
    printf("%-34s %.1f GiB  (main memory)\n", "global memory", prop.totalGlobalMem / 1073741824.0);
    printf("%-34s %.0f GB/s  (bus %d bit, %d MHz, double data rate)\n", "memory bandwidth (peak)",
           2.0 * memClock * 1e3 * (busWidth / 8) / 1e9, busWidth, memClock / 1000);
    printf("%-34s %d MHz\n", "clock (max.)", clock / 1000);
    printf("%-34s %d  (hardware threads of the CPU)\n", "threads in flight (max.)", sms * threadsSM);

    // the same numbers in the form used by the Python cells of this notebook
    printf("\nfor the Little's-law cell (replace the latency by your measurement):\n");
    printf("    'my GPU': dict(bandwidth=%.0fe9, latency=500e-9, units=%d, threads=%d),\n",
           2.0 * memClock * 1e3 * (busWidth / 8) / 1e9, sms, sms * threadsSM);
    printf("for the occupancy calculator:\n");
    printf("    limits = dict(max_warps=%d, max_blocks=%d, regs_per_sm=%d, smem_per_sm=%d)\n",
           threadsSM / warp, blocksSM, regsSM, smemSM);
    return 0;
}

> 🤔 Compare the output with the B200 numbers in the table above, and with the Little's-law estimate: how many threads does your GPU keep resident, and how many bytes must each of them have in flight?
>
> The last lines of the output have the form of the Python cells: paste the first into the Little's-law cell above to repeat the estimate for your GPU, and the second into the occupancy calculator below.

## The CUDA Programming Model

### Kernels and the Thread Hierarchy

A **kernel** is a function marked `__global__`: called from the host, executed on the device by every thread of a **grid**.
The grid consists of **blocks** of up to 1024 threads; the hardware executes the threads of a block in **warps** of 32.

![The CUDA thread hierarchy: a grid of blocks, each block made of warps of 32 threads; the block scheduler assigns blocks to SMs as resources become free, the remaining blocks wait](img/gpu-thread-hierarchy.svg)

```cpp
__global__ void kernel(float *x, int n) {           // runs on the GPU, called from the host
    int i = blockIdx.x * blockDim.x + threadIdx.x;  // global index of this thread
    if (i < n)
        x[i] = …;
}

kernel<<<blocks, threads>>>(d_x, n);                // launch a grid of blocks × threads threads
```

Every thread finds its piece of work through built-in variables: `threadIdx` (its index in the block), `blockIdx` (the index of its block), `blockDim` and `gridDim` (the sizes).
This is the same idea as an MPI rank or `omp_get_thread_num()` — **SPMD** again, now with millions of instances.
Functions called from kernels are marked `__device__`.

> 🤔 **Predict:** The following kernel prints one line per warp. In which order do the lines appear — and does `host: kernel launched` come before or after them?

In [ ]:
%%cuda -r -i cuda-check.h -f Icode -o code/cuda-hello.cu
// the number of the SM this thread runs on: a special register, read with inline PTX assembly
__device__ unsigned sm_id(void) {
    unsigned id;
    asm volatile("mov.u32 %0, %%smid;" : "=r"(id));
    return id;
}

// a kernel: executed by every thread of the grid
__global__ void hello(void) {
    unsigned warp = threadIdx.x / warpSize, lane = threadIdx.x % warpSize;
    if (lane == 0)                              // one line per warp
        printf("block %u, warp %u (threads %3u-%3u) runs on SM %u\n",
               blockIdx.x, warp, threadIdx.x, threadIdx.x + warpSize - 1, sm_id());
}

int main(void) {
    hello<<<4, 96>>>();                         // a grid of 4 blocks of 96 threads (3 warps) each
    CUDA_CHECK_LAUNCH();
    printf("host: kernel launched\n");
    CUDA_CHECK(cudaDeviceSynchronize());        // wait for the kernel - this also flushes its output
    printf("host: kernel finished\n");
    return 0;
}

* The kernel launch is **asynchronous**: the host continues immediately, like after `MPI_Isend()`. `cudaDeviceSynchronize()` waits for the GPU — the counterpart of `MPI_Wait()`.
* The blocks run on different SMs, in no particular order. The hardware **block scheduler** assigns a block to an SM as soon as the SM has room for it, and the block stays there until it is finished.
* Blocks must therefore be **independent**: a kernel has no barrier across blocks, and a block must never wait for another one — it might not even have started.
  It is the same rule as for the iterations of an OpenMP `parallel for`, and the block scheduler acts like `schedule(dynamic)` with one block as chunk.

### Memory: Two Address Spaces

Host and device have **separate memories**: device memory is allocated with `cudaMalloc()`, and data is copied explicitly with `cudaMemcpy()` — very much like an MPI message to a process on another node.

```cpp
double *d_x;                                                // "d_": a device pointer — not valid on the host
cudaMalloc(&d_x, bytes);
cudaMemcpy(d_x, x, bytes, cudaMemcpyHostToDevice);          // host → device
kernel<<<blocks, threads>>>(d_x, n);
cudaMemcpy(x, d_x, bytes, cudaMemcpyDeviceToHost);          // device → host (waits for the kernel)
cudaFree(d_x);
```

*Unified memory* (`cudaMallocManaged()`) gives one pointer valid on both sides, with pages migrated on demand; on systems like Grace Hopper, the CPU and GPU even share a coherent address space.
Convenient — but the data still has to move, and you should know when it does.

Here is the vector triad `a[i] = b[i] + s * c[i]` — the STREAM benchmark kernel — with one thread per element.
The times are measured with **events**: markers in the GPU's command stream that the GPU timestamps when it reaches them, see `GpuTimer` in the header.
Host timers would only measure the asynchronous launch.

> 🤔 **Predict:** How do the times of the copies compare with the time of the kernel?

In [ ]:
%%cuda -r -i cuda-check.h -f Icode -o code/cuda-triad.cu
// one thread per element: the loop body of a[i] = b[i] + s * c[i]
__global__ void triad(double *a, const double *b, const double *c, double s, int64_t n) {
    int64_t i = blockIdx.x * (int64_t)blockDim.x + threadIdx.x;     // global thread index
    if (i < n)                                  // the last block may stick out of the array
        a[i] = b[i] + s * c[i];
}

int main(void) {
    const int64_t n = 1 << 25;                  // 32 Mi elements: 256 MiB per array
    const size_t bytes = n * sizeof(double);

    // host memory: ordinary malloc
    double *a = (double *)malloc(bytes), *b = (double *)malloc(bytes), *c = (double *)malloc(bytes);
    for (int64_t i = 0; i < n; ++i) {
        a[i] = 0.0;                             // touch a, too: the copy back should not measure page faults
        b[i] = 1.0;
        c[i] = 2.0;
    }

    // device memory: a separate address space - d_a cannot be dereferenced on the host
    double *d_a, *d_b, *d_c;
    CUDA_CHECK(cudaMalloc(&d_a, bytes));
    CUDA_CHECK(cudaMalloc(&d_b, bytes));
    CUDA_CHECK(cudaMalloc(&d_c, bytes));

    GpuTimer timer;
    timer.start();
    CUDA_CHECK(cudaMemcpy(d_b, b, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_c, c, bytes, cudaMemcpyHostToDevice));
    float ms_in = timer.stop();

    const int threads = 256;                    // threads per block
    const int64_t blocks = (n + threads - 1) / threads;
    triad<<<blocks, threads>>>(d_a, d_b, d_c, 3.0, n);      // warm-up: the first launch has extra costs
    CUDA_CHECK_LAUNCH();

    const int reps = 10;
    timer.start();
    for (int r = 0; r < reps; ++r)
        triad<<<blocks, threads>>>(d_a, d_b, d_c, 3.0, n);
    float ms_kernel = timer.stop() / reps;

    timer.start();
    CUDA_CHECK(cudaMemcpy(a, d_a, bytes, cudaMemcpyDeviceToHost));
    float ms_out = timer.stop();

    int64_t wrong = 0;
    for (int64_t i = 0; i < n; ++i)
        wrong += a[i] != 7.0;
    printf("%lld blocks of %d threads, %lld wrong results\n", (long long)blocks, threads, (long long)wrong);
    printf("copy b, c to the GPU: %7.2f ms  %7.1f GB/s\n", ms_in, 2 * bytes / ms_in / 1e6);
    printf("triad kernel        : %7.2f ms  %7.1f GB/s\n", ms_kernel, 3 * bytes / ms_kernel / 1e6);
    printf("copy a to the host  : %7.2f ms  %7.1f GB/s\n", ms_out, bytes / ms_out / 1e6);

    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_c));
    free(a);
    free(b);
    free(c);
    return 0;
}

For comparison, the same triad on one CPU core (this cell runs everywhere):

In [ ]:
%%c -r -o code/cuda-cpu-triad.c
#include <stdio.h>
#include <stdlib.h>
#include <stdint.h>
#include <time.h>

// wall-clock time in seconds; timespec_get() is standard C11, unlike POSIX clock_gettime()
static double seconds(void) {
    struct timespec t;
    timespec_get(&t, TIME_UTC);
    return t.tv_sec + 1e-9 * t.tv_nsec;
}

int main(void) {
    const int64_t n = 1 << 25;                  // the same size as on the GPU: 256 MiB per array
    const size_t bytes = n * sizeof(double);
    double *a = malloc(bytes), *b = malloc(bytes), *c = malloc(bytes);
    for (int64_t i = 0; i < n; ++i) {
        a[i] = 0.0;
        b[i] = 1.0;
        c[i] = 2.0;
    }

    const int reps = 10;
    double t0 = seconds();
    for (int r = 0; r < reps; ++r) {
        for (int64_t i = 0; i < n; ++i)
            a[i] = b[i] + 3.0 * c[i];
        __asm__ volatile("" : : "r"(a) : "memory");     // keeps the compiler from dropping repetitions
    }
    double t = (seconds() - t0) / reps;

    printf("triad on one core: %.2f ms, %.1f GB/s (a[0] = %.1f)\n", 1e3 * t, 3 * bytes / t / 1e9, a[0]);
    free(a);
    free(b);
    free(c);
    return 0;
}

* The kernel reaches a large fraction of the GPU's memory bandwidth — far more than one CPU core, and several times what a whole CPU socket can deliver.
* The copies over PCIe are **much slower than the kernel**: moving the data to the GPU costs more than the computation on it.
  The first rule of GPU programming: **keep the data on the GPU**, and copy only what is needed — like minimizing communication in MPI.
* Note what the GPU version *doesn't* have: no loop. The loop has become the grid, and the loop index the thread index. The `if (i < n)` is needed because the grid consists of whole blocks.
* On the CPU, the triad moves 32 instead of 24 bytes per element if the cache first loads the line of `a` that is about to be overwritten (*write-allocate*) — the GB/s shown counts 24.

## Execution: Warps, SIMT and SIMD

A warp executes **one instruction for all its 32 threads at a time** — it is a SIMD machine.
NVIDIA calls it **SIMT** (single instruction, multiple threads), and the difference to CPU SIMD is in the programming model, not in the hardware:

| | CPU SIMD (AVX-512) | GPU SIMT |
|---|---|---|
| code written as | vector instructions — by the compiler (auto-vectorization), intrinsics or `#pragma omp simd` | scalar code for one thread |
| vector width | visible: 8 doubles / 16 floats per register | hidden: 32 threads per warp |
| branches | the compiler generates both paths and blends them with mask registers | the hardware masks inactive threads |
| non-contiguous loads | gather instructions, slow | allowed per thread — but see coalescing below |
| if vectorization fails | the loop runs scalar, 8–16× slower | not possible: the code is always "vectorized" |

The scalar programming model is convenient, but the vector hardware is still there — and it shows up in two places: branches and memory accesses.

### Branch Divergence

![Branch divergence: when the threads of a warp take different paths, the warp executes both paths one after the other with the inactive lanes masked off; a CPU vectorizes the same branch with mask registers and also computes both sides; if whole warps take the same path, there is no extra cost](img/gpu-divergence.svg)

If the threads of a warp take different paths, the warp executes **all taken paths one after the other**, with the threads of the other path masked off.
The cost is the sum of the paths — exactly as for a branch in a vectorized CPU loop.
If all threads *of a warp* take the same path, there is no penalty, even if different warps take different paths.
(Since the Volta generation, every thread has its own program counter — *independent thread scheduling* — so that divergent threads can't deadlock each other, but a warp still executes one path at a time.)

Note what a GPU does **not** have: branch prediction and speculation. A CPU core guesses the outcome of every branch and runs ahead; a mispredicted branch costs it 15–20 cycles of discarded work. A GPU simply waits for the branch — and runs another warp in the meantime.

> 🤔 **Predict:** Paths A and B take times $t_A$ and $t_B$. What times do you expect for "warps alternate" and for "threads alternate"?

In [ ]:
%%cuda -r -i cuda-check.h -f Icode -o code/cuda-divergence.cu
// two code paths with different instructions, so that the compiler cannot merge them
__device__ float path_a(float x, int k) {
    for (int i = 0; i < k; ++i)
        x = x * 0.999f + 0.5f;
    return x;
}
__device__ float path_b(float x, int k) {
    for (int i = 0; i < k; ++i)
        x = sqrtf(x + 1.0f);
    return x;
}

// mode 0: all threads take path A, mode 1: all take path B,
// mode 2: whole warps alternate between A and B, mode 3: neighboring threads alternate
__global__ void branches(float *x, int mode, int k) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    bool a = mode == 0 ? true
           : mode == 1 ? false
           : mode == 2 ? (i / 32) % 2 == 0
           :             i % 2 == 0;
    if (a)
        x[i] = path_a(x[i], k);
    else
        x[i] = path_b(x[i], k);
}

int main(void) {
    const int n = 1 << 22, k = 2000;
    float *x = (float *)malloc(n * sizeof(float)), *d_x;
    for (int i = 0; i < n; ++i)
        x[i] = 1.0f;
    CUDA_CHECK(cudaMalloc(&d_x, n * sizeof(float)));
    CUDA_CHECK(cudaMemcpy(d_x, x, n * sizeof(float), cudaMemcpyHostToDevice));

    const char *name[] = { "all threads path A", "all threads path B",
                           "warps alternate A/B", "threads alternate A/B" };
    GpuTimer timer;
    branches<<<n / 256, 256>>>(d_x, 0, k);      // warm-up
    CUDA_CHECK_LAUNCH();
    for (int mode = 0; mode < 4; ++mode) {
        timer.start();
        branches<<<n / 256, 256>>>(d_x, mode, k);
        float ms = timer.stop();
        CUDA_CHECK_LAUNCH();
        printf("%-22s: %6.2f ms\n", name[mode], ms);
    }

    CUDA_CHECK(cudaFree(d_x));
    free(x);
    return 0;
}

With alternating warps, the time is about the average of $t_A$ and $t_B$: half of the warps take A, the other half B, and the SM runs them concurrently.
With alternating threads, every warp executes both paths — the time is about $t_A + t_B$, and half of the lanes are idle.
**Divergence is about warps, not threads**: arrange the work so that the threads of a warp take the same branch whenever you can.

### Memory Coalescing

The load/store unit combines the 32 addresses of a warp's memory instruction into as few memory transactions as possible.
Memory is accessed in **32-byte sectors** (four per 128-byte cache line).
If the 32 threads read 32 consecutive floats, that's 128 bytes — four sectors, every byte used.
With a stride, the same 32 loads touch more sectors, and most of the transferred bytes are wasted:

![Memory coalescing: the 32 loads of a warp are combined into accesses of 32-byte sectors; consecutive addresses need 4 sectors, a stride of 2 needs 8 sectors of which half is wasted, a stride of 8 or more needs 32 sectors of which only 4 bytes each are used](img/gpu-coalescing.svg)

This is the GPU's version of something you know from the CPU: memory is transferred in **cache lines** (64 bytes on x86, 128 bytes on Apple M processors), whether you use all of their bytes or not; and a SIMD load needs contiguous data.
The **rule for both: consecutive threads (lanes) should access consecutive addresses**.

> 🤔 **Predict:** The following kernels copy every `stride`-th float, the CPU cell sums every `stride`-th `int64_t`. How does the time change with the stride, and up to which stride?

In [ ]:
%%cuda -r -i cuda-check.h -f Icode -o code/cuda-stride.cu
// thread t copies element t * stride
__global__ void strided_copy(float *out, const float *in, int stride, int64_t n) {
    int64_t i = (blockIdx.x * (int64_t)blockDim.x + threadIdx.x) * stride;
    if (i < n)
        out[i] = in[i];
}

int main(void) {
    const int64_t n = 1 << 26;                  // 256 MiB per array
    float *in, *out;
    CUDA_CHECK(cudaMalloc(&in, n * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&out, n * sizeof(float)));
    CUDA_CHECK(cudaMemset(in, 0, n * sizeof(float)));

    GpuTimer timer;
    strided_copy<<<n / 256, 256>>>(out, in, 1, n);         // warm-up
    CUDA_CHECK_LAUNCH();
    printf("stride   elements  time [ms]  useful GB/s\n");
    for (int stride = 1; stride <= 64; stride *= 2) {
        const int64_t count = n / stride;        // elements copied
        timer.start();
        strided_copy<<<(count + 255) / 256, 256>>>(out, in, stride, n);
        float ms = timer.stop();
        CUDA_CHECK_LAUNCH();
        printf("%6d  %9lld  %9.3f  %11.1f\n", stride, (long long)count, ms, 2 * count * sizeof(float) / ms / 1e6);
    }

    CUDA_CHECK(cudaFree(in));
    CUDA_CHECK(cudaFree(out));
    return 0;
}

In [ ]:
%%c -r -o code/cuda-cpu-stride.c
#include <stdio.h>
#include <stdlib.h>
#include <stdint.h>
#include <time.h>

// wall-clock time in seconds; timespec_get() is standard C11, unlike POSIX clock_gettime()
static double seconds(void) {
    struct timespec t;
    timespec_get(&t, TIME_UTC);
    return t.tv_sec + 1e-9 * t.tv_nsec;
}

int main(void) {
    const int64_t n = 1 << 25;                  // 256 MiB
    int64_t *a = malloc(n * sizeof(int64_t));
    for (int64_t i = 0; i < n; ++i)
        a[i] = i;

    printf("stride   elements  time [ms]  useful GB/s\n");
    for (int stride = 1; stride <= 64; stride *= 2) {
        int64_t sum = 0;
        double best = 1e30;
        for (int r = 0; r < 5; ++r) {           // the fastest of 5 runs
            double t0 = seconds();
            if (stride == 1)                    // a separate loop with a constant stride can be vectorized
                for (int64_t i = 0; i < n; ++i)
                    sum += a[i];
            else
                for (int64_t i = 0; i < n; i += stride)
                    sum += a[i];
            double t = seconds() - t0;
            best = t < best ? t : best;
        }
        const int64_t count = n / stride;       // elements read
        printf("%6d  %9lld  %9.2f  %11.2f   (sum %lld)\n", stride, (long long)count, 1e3 * best,
               count * sizeof(int64_t) / best / 1e9, (long long)sum);
    }
    free(a);
    return 0;
}

On both processors, the **time stays about the same** as long as every sector or cache line still contains an element that is needed — although fewer and fewer elements are read.
The useful bandwidth drops by the stride.
Only beyond the transfer unit (8 floats per 32-byte sector on the GPU; 8 `int64_t` per 64-byte cache line on the CPU), the time starts to drop as well — on the CPU possibly only from stride 32 on, since the prefetchers also fetch neighboring lines.
(The CPU cell has a separate loop for stride 1: only a loop with a constant stride can be vectorized — the CPU's own version of coalescing. Without it, the scalar loop would manage one addition per cycle, too slow to keep up with the memory.)

In multidimensional arrays, this decides which index is mapped to `threadIdx.x`: the one that is contiguous in memory — the index of the innermost loop on the CPU.

## Scheduling and Latency Hiding

Now to the central difference.
Both processors must keep their execution units busy while instructions wait for memory — but they find the independent work to do in different places.

| | CPU | GPU |
|---|---|---|
| **work to cores** | the OS scheduler maps software threads to hardware threads: preemptive, time slices of milliseconds; the OpenMP runtime distributes loop iterations | the hardware **block scheduler** assigns blocks to SMs as resources become free; blocks run to completion, in any order |
| **context switch** | OS: microseconds, registers saved to memory; SMT: in hardware, 2 contexts | **zero cost**: the registers of all resident warps stay in the register file |
| **instruction scheduling** | **out of order**: from a window of 300–500 instructions of 1–2 threads, any instruction whose operands are ready; speculation beyond branches | **in order per warp**: every cycle, each of the 4 warp schedulers issues the next instruction of *one* of its ready (*eligible*) warps |
| **independent work from** | the **same** thread: instruction-level parallelism (ILP), several cache misses in flight | **other** warps: thread-level parallelism (TLP) — plus ILP within a thread |
| **latency is** | avoided: large caches, prefetchers; hidden: out-of-order execution, SMT | hidden: many warps; avoided: shared memory, caches |

![Latency hiding: a CPU core overlaps a cache miss with the independent instructions of the same thread until its reorder buffer is full and then stalls, SMT adds a second thread; a GPU warp scheduler switches every cycle to another ready warp, so with enough warps the issue slot is never idle](img/gpu-latency-hiding.svg)

SMT on a CPU — Intel's Hyper-Threading — is exactly the GPU's idea, just with 2 threads instead of 64 warps per core.
The GPU can afford so many because a warp's context is cheap: no reorder buffer, no rename tables, just registers.

### Latency Hiding on the CPU: ILP

Let's first watch the CPU do it.
A sum has a **dependency chain**: every addition needs the previous result and waits for the latency of the floating-point adder (3–4 cycles).
With several independent partial sums, the out-of-order engine can have several additions in flight.
The cell is compiled without vectorization, so that every addition is a scalar instruction and only ILP is at work.

> 🤔 **Predict:** How much faster are two, four and eight chains? And why are the sums not exactly equal?

In [ ]:
%%c -r -f fno-tree-vectorize fno-tree-slp-vectorize -o code/cuda-cpu-ilp.c
#include <stdio.h>
#include <stdlib.h>
#include <time.h>

// wall-clock time in seconds; timespec_get() is standard C11, unlike POSIX clock_gettime()
static double seconds(void) {
    struct timespec t;
    timespec_get(&t, TIME_UTC);
    return t.tv_sec + 1e-9 * t.tv_nsec;
}

// the sum of a, reps times, with 1 to 8 independent partial sums (dependency chains):
// an addition in one chain does not need the results of the other chains.
// Inlined with a constant number of chains, so that the partial sums stay in registers.
static inline __attribute__((always_inline)) double sum(const double *a, int n, int reps, int chains) {
    double p[8] = { 0.0 };
    for (int r = 0; r < reps; ++r)
        for (int i = 0; i < n; i += chains)
            for (int k = 0; k < chains; ++k)
                p[k] += a[i + k];
    return p[0] + p[1] + p[2] + p[3] + p[4] + p[5] + p[6] + p[7];
}

int main(void) {
    const int n = 2048, reps = 100000;          // 16 KiB: stays in the L1 cache, no memory latency involved
    double *a = malloc(n * sizeof(double));
    for (int i = 0; i < n; ++i)
        a[i] = 1.0 / (i + 1);

    printf("chains  ns per addition  sum\n");
    for (int chains = 1; chains <= 8; chains *= 2) {
        double t0 = seconds(), s;
        switch (chains) {
            case 1:  s = sum(a, n, reps, 1); break;     // every addition waits for the previous one
            case 2:  s = sum(a, n, reps, 2); break;
            case 4:  s = sum(a, n, reps, 4); break;
            default: s = sum(a, n, reps, 8);            // up to eight additions in flight
        }
        double t = seconds() - t0;
        printf("%6d  %15.3f  %.10f\n", chains, 1e9 * t / ((double)n * reps), s);
    }
    free(a);
    return 0;
}

With one chain, an addition takes the adder's latency.
With more chains, the core overlaps the additions: the time per addition drops with the number of chains, until all floating-point adders of the core are busy in every cycle.
Independent chains are also what a compiler needs for SIMD: remove the two `-f` flags, and it may vectorize the partial sums on top.
The results differ in the last digits, since floating-point addition is not associative — which is also why the compiler may not do this transformation for you without `-ffast-math`.
(You met the same effect with `MPI_Reduce()` in Collective Communication.)

The same works for memory, with far larger latencies.
The next cell performs the same random loads twice: first as a **pointer chase**, where the address of each load is the result of the previous one, then with all addresses known in advance:

In [ ]:
%%c -r -o code/cuda-cpu-chase.c
#include <stdio.h>
#include <stdlib.h>
#include <stdint.h>
#include <time.h>

// wall-clock time in seconds; timespec_get() is standard C11, unlike POSIX clock_gettime()
static double seconds(void) {
    struct timespec t;
    timespec_get(&t, TIME_UTC);
    return t.tv_sec + 1e-9 * t.tv_nsec;
}

int main(void) {
    const uint32_t n = 1u << 25;                // 128 MiB: much larger than the caches
    const int64_t steps = 1 << 23;

    // a random-looking cycle through all n elements: next[i] = (a * i + c) mod n is a single cycle
    // for n = 2^k, odd c and a = 1 mod 4 (a full-period linear congruential generator)
    uint32_t *next = malloc(n * sizeof(uint32_t));
    for (uint32_t i = 0; i < n; ++i)
        next[i] = (1664525u * i + 1013904223u) & (n - 1);

    // the addresses visited by the chain, recorded in advance
    uint32_t *addr = malloc(steps * sizeof(uint32_t)), p = 0;
    for (int64_t s = 0; s < steps; ++s) {
        addr[s] = p;
        p = next[p];
    }

    // dependent loads: the address of the next load is the result of this one
    double t0 = seconds();
    p = 0;
    for (int64_t s = 0; s < steps; ++s)
        p = next[p];
    double t1 = seconds();

    // the same loads, but independent: all addresses are known in advance
    uint64_t sum = 0;
    for (int64_t s = 0; s < steps; ++s)
        sum += next[addr[s]];
    double t2 = seconds();

    printf("dependent loads  : %6.2f ns per load (p = %u)\n", 1e9 * (t1 - t0) / steps, p);
    printf("independent loads: %6.2f ns per load (sum = %llu)\n", 1e9 * (t2 - t1) / steps, (unsigned long long)sum);
    free(next);
    free(addr);
    return 0;
}

The dependent loads expose the full memory latency of about 100 ns.
The independent loads are more than an order of magnitude faster: the out-of-order core issues loads far ahead and keeps many cache misses in flight at the same time (*memory-level parallelism*) — all from **one** thread.

### Latency Hiding on the GPU: Many Warps

Now the same on the GPU.
First, one single thread chases pointers — a pure latency measurement.
Then every thread of a growing number of warps per SM chases its own chain.
A thread can't do anything while its load is on the way — but the SM can switch to another warp.

> 🤔 **Predict:** How does the latency of a single GPU thread compare with the CPU's? How should the throughput grow with the number of warps per SM?

In [ ]:
%%cuda -r -i cuda-check.h -f Icode -o code/cuda-latency.cu
// a random-looking cycle through all n elements: next[i] = (a * i + c) mod n is a single cycle
// for n = 2^k, odd c and a = 1 mod 4 (a full-period linear congruential generator)
__global__ void init(uint32_t *next, uint32_t n) {
    for (uint32_t i = blockIdx.x * blockDim.x + threadIdx.x; i < n; i += gridDim.x * blockDim.x)
        next[i] = (1664525u * i + 1013904223u) & (n - 1);
}

// a hash: random starting points for the threads
__device__ uint32_t scramble(uint32_t x) {
    x ^= x >> 16;
    x *= 0x45d9f3bu;
    x ^= x >> 16;
    return x;
}

// every thread follows its own chain of dependent loads: the address of a load is the result of the previous one
__global__ void chase(const uint32_t *next, uint32_t n, int steps, uint32_t *out) {
    uint32_t tid = blockIdx.x * blockDim.x + threadIdx.x;
    uint32_t p = scramble(tid) & (n - 1);
    for (int s = 0; s < steps; ++s)
        p = next[p];
    out[tid] = p;                               // keeps the compiler from removing the loop
}

int main(void) {
    int device, threadsSM, l2;
    CUDA_CHECK(cudaGetDevice(&device));
    CUDA_CHECK(cudaDeviceGetAttribute(&threadsSM, cudaDevAttrMaxThreadsPerMultiProcessor, device));
    CUDA_CHECK(cudaDeviceGetAttribute(&l2, cudaDevAttrL2CacheSize, device));
    const int sms = sm_count(), max_warps = threadsSM / 32;

    // the table must be much larger than the L2 cache, so that almost every load goes to memory:
    // at least 256 MiB, doubled up to 16x the L2 cache (but at most a quarter of the free memory)
    size_t free_mem, total_mem;
    CUDA_CHECK(cudaMemGetInfo(&free_mem, &total_mem));
    uint32_t n = 1u << 26;                      // elements of 4 bytes, a power of two
    while ((size_t)n * 4 < 16 * (size_t)l2 && (size_t)n * 8 <= free_mem / 4 && n < (1u << 30))
        n *= 2;
    printf("table: %.0f MiB, L2 cache: %.0f MiB\n", n * 4.0 / 1048576, l2 / 1048576.0);

    uint32_t *next, *out;
    CUDA_CHECK(cudaMalloc(&next, n * sizeof(uint32_t)));
    CUDA_CHECK(cudaMalloc(&out, (size_t)sms * threadsSM * sizeof(uint32_t)));
    init<<<1024, 256>>>(next, n);
    CUDA_CHECK_LAUNCH();

    // 1. a single thread: every load waits for the previous one - pure latency
    GpuTimer timer;
    const int steps1 = 100000;
    chase<<<1, 1>>>(next, n, 1000, out);        // warm-up
    CUDA_CHECK_LAUNCH();
    timer.start();
    chase<<<1, 1>>>(next, n, steps1, out);
    const double latency = timer.stop() * 1e6 / steps1;     // ns per load
    printf("1 thread: %.0f ns per dependent load\n\n", latency);

    // 2. w warps per SM, every thread with its own chain: the SM switches to another warp
    //    whenever one waits for its load. Every thread has one load in flight at any time, and
    //    Little's law gives the latency under load: loads in flight / throughput
    const int steps = 2000;
    printf("warps/SM  loads in flight (= threads)  loads/ns  latency [ns]\n");
    for (int w = 1; w <= max_warps; w *= 2) {
        const int threads = w == 1 ? 32 : 64;   // threads per block
        const int blocks = sms * w * 32 / threads;
        timer.start();
        chase<<<blocks, threads>>>(next, n, steps, out);
        const float ms = timer.stop();
        const double rate = (double)blocks * threads * steps / (ms * 1e6);
        printf("%8d  %28d  %8.1f  %12.0f\n", w, blocks * threads, rate, blocks * threads / rate);
    }

    CUDA_CHECK(cudaFree(next));
    CUDA_CHECK(cudaFree(out));
    return 0;
}

* A **single GPU thread is slow**: its memory latency is several times the CPU's, and it can't overlap anything, since it has no out-of-order engine.
* The throughput grows **linearly with the number of warps** — every warp adds another 32 loads in flight — as long as the latency in the last column stays the same: the memory system serves all these loads concurrently.
  This is Little's law in action: throughput = loads in flight / latency.
  Once the memory system saturates, the throughput stays constant, and more warps only make every load wait longer.
* Even with one warp per SM, the GPU has thousands of loads in flight, and with all warps hundreds of thousands — far beyond what any CPU could have pending.
* The CPU hides latency by looking *ahead in one thread*; the GPU hides it by looking *across many threads*.
  Neither works for free: the CPU needs independent instructions in the thread, the GPU needs enough threads.

### Occupancy: the Price of Registers

How many warps can be resident on an SM?
The SM's resources are **partitioned statically** among the resident blocks when they start:

* at most 64 warps (2048 threads) and 32 blocks per SM,
* 65 536 registers per SM: a kernel using 64 registers per thread fits 1024 threads, one with 128 registers only 512,
* 228 KiB of shared memory per SM, divided among the blocks.

These are the limits of the H100 and B200. Other GPUs differ: an A100 has 164 KiB of shared memory per SM, an A40 only 48 warps, 16 blocks and 100 KiB — the device query prints the limits of your GPU.

The ratio of resident warps to the maximum is the **occupancy**.
On a CPU, the register file is a fixed property of the hardware thread; on a GPU, the **number of registers per thread is a trade-off** made by the compiler: more registers per thread mean fewer resident warps, and less latency hiding.
`-f Xptxas=-v` makes `nvcc` report the registers used per thread, `__launch_bounds__` or `-maxrregcount` limit them.

Here is a simplified occupancy calculator; replace `limits` by the line of the device query to use the limits of your GPU:

In [ ]:
from math import ceil

# the limits of an H100 or B200 - the device query prints this line for your GPU
limits = dict(max_warps=64, max_blocks=32, regs_per_sm=65536, smem_per_sm=228 * 1024)

def occupancy(threads_per_block, regs_per_thread, smem_per_block=0, *,
              max_warps, max_blocks, regs_per_sm, smem_per_sm):
    """simplified occupancy calculation for the given limits of an SM:
    returns the resident blocks per SM, the occupancy and the limiting resource"""
    warps = ceil(threads_per_block / 32)
    regs_per_warp = ceil(regs_per_thread * 32 / 256) * 256        # allocated in units of 256 registers per warp
    fits = {                                                      # blocks that fit, per resource
        'warps':         max_warps // warps,
        'blocks':        max_blocks,
        'registers':     regs_per_sm // (regs_per_warp * warps),
        'shared memory': smem_per_sm // smem_per_block if smem_per_block else max_blocks,
    }
    limiter = min(fits, key=fits.get)
    return fits[limiter], fits[limiter] * warps / max_warps, limiter

print('threads/block  registers/thread  shared memory/block  blocks/SM  occupancy  limited by')
for tpb, regs, smem in [(256, 32, 0), (256, 64, 0), (256, 128, 0), (256, 255, 0),
                        (32, 32, 0), (1024, 32, 0), (256, 32, 48 * 1024)]:
    blocks, occ, limiter = occupancy(tpb, regs, smem, **limits)
    print(f'{tpb:13d}  {regs:16d}  {smem // 1024:15d} KiB  {blocks:9d}  {occ:9.0%}  {limiter}')

The CUDA runtime does the exact calculation for a compiled kernel with `cudaOccupancyMaxActiveBlocksPerMultiprocessor()`:

In [ ]:
%%cuda -r -i cuda-check.h -f Icode -o code/cuda-occupancy.cu
__global__ void triad(double *a, const double *b, const double *c, double s, int64_t n) {
    int64_t i = blockIdx.x * (int64_t)blockDim.x + threadIdx.x;
    if (i < n)
        a[i] = b[i] + s * c[i];
}

int main(void) {
    int device, threadsSM;
    CUDA_CHECK(cudaGetDevice(&device));
    CUDA_CHECK(cudaDeviceGetAttribute(&threadsSM, cudaDevAttrMaxThreadsPerMultiProcessor, device));
    const int max_warps = threadsSM / 32;

    // the resources of the compiled kernel
    cudaFuncAttributes attr;
    CUDA_CHECK(cudaFuncGetAttributes(&attr, triad));
    printf("triad: %d registers per thread, %zu bytes of shared memory per block\n\n", attr.numRegs, attr.sharedSizeBytes);

    // how many blocks fit on one SM, depending on the block size and the shared memory per block?
    printf("threads/block  shared memory/block  blocks/SM  warps/SM  occupancy\n");
    const size_t smem[] = { 0, 16 * 1024, 48 * 1024 };
    for (int s = 0; s < 3; ++s)
        for (int threads = 32; threads <= 1024; threads *= 4) {
            int blocks;
            CUDA_CHECK(cudaOccupancyMaxActiveBlocksPerMultiprocessor(&blocks, triad, threads, smem[s]));
            const int warps = blocks * threads / 32;
            printf("%13d  %15zu KiB  %9d  %8d  %8.0f %%\n", threads, smem[s] / 1024, blocks, warps, 100.0 * warps / max_warps);
        }
    return 0;
}

Blocks of 32 threads can't fill an SM, since the number of blocks per SM is limited; use **128–512 threads per block** as a starting point.

Is full occupancy necessary? No — **ILP works on the GPU, too.**
If every thread has several independent loads in flight, fewer warps suffice to cover the latency, and each thread can use more registers.
The next cell copies an array with a limited number of warps per SM, issuing 1, 4 or 8 independent loads per thread before the first store:

> 🤔 **Predict:** With how many warps per SM does each variant reach the full bandwidth?

In [ ]:
%%cuda -r -i cuda-check.h -f Icode -o code/cuda-ilp.cu
// copy with ILP independent loads per thread and loop iteration: all ILP loads are issued
// before the first result is needed - like the independent loads of an out-of-order CPU core
template <int ILP>
__global__ void copy(float *out, const float *in, int64_t n) {
    const int64_t stride = (int64_t)gridDim.x * blockDim.x;
    for (int64_t i = blockIdx.x * (int64_t)blockDim.x + threadIdx.x; i < n; i += ILP * stride) {
        float v[ILP];
        #pragma unroll
        for (int k = 0; k < ILP; ++k)
            if (i + k * stride < n)
                v[k] = in[i + k * stride];
        #pragma unroll
        for (int k = 0; k < ILP; ++k)
            if (i + k * stride < n)
                out[i + k * stride] = v[k];
    }
}

// bandwidth in GB/s of the copy with a grid of w warps per SM
template <int ILP>
double bandwidth(float *out, const float *in, int64_t n, int sms, int w) {
    const int threads = w == 1 ? 32 : 64;
    const int blocks = sms * w * 32 / threads;
    GpuTimer timer;
    timer.start();
    copy<ILP><<<blocks, threads>>>(out, in, n);
    const float ms = timer.stop();
    CUDA_CHECK_LAUNCH();
    return 2.0 * n * sizeof(float) / ms / 1e6;
}

int main(void) {
    const int64_t n = 1 << 26;                  // 256 MiB per array
    int device, threadsSM;
    CUDA_CHECK(cudaGetDevice(&device));
    CUDA_CHECK(cudaDeviceGetAttribute(&threadsSM, cudaDevAttrMaxThreadsPerMultiProcessor, device));
    const int sms = sm_count(), max_warps = threadsSM / 32;

    float *in, *out;
    CUDA_CHECK(cudaMalloc(&in, n * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&out, n * sizeof(float)));
    CUDA_CHECK(cudaMemset(in, 0, n * sizeof(float)));
    bandwidth<1>(out, in, n, sms, max_warps);   // warm-up

    printf("warps/SM   GB/s with ILP = 1   ILP = 4   ILP = 8\n");
    for (int w = 1; w <= max_warps; w *= 2)
        printf("%8d  %18.0f  %8.0f  %8.0f\n", w, bandwidth<1>(out, in, n, sms, w),
               bandwidth<4>(out, in, n, sms, w), bandwidth<8>(out, in, n, sms, w));

    CUDA_CHECK(cudaFree(in));
    CUDA_CHECK(cudaFree(out));
    return 0;
}

With 4 or 8 loads per thread, a fraction of the warps reaches the same bandwidth as one load per thread with full occupancy — this is how high-performance libraries often run at low occupancy.
It also explains why the triad of the first measurement gets away with one element per thread: every thread has *two* independent loads in flight, `b[i]` and `c[i]` — 16 bytes, where the Little's-law estimate for the B200 asked for 13.
Both processors balance the same two sources of independent work; they just weight them differently:
the CPU core relies on ILP and has little TLP (SMT), the GPU relies on TLP and can use ILP in addition.

## Cooperation Within a Block: Shared Memory and Synchronization

The threads of a block run on the same SM and can cooperate:

* **`__shared__` variables** live in the SM's shared memory and are visible to all threads of the block — a fast, explicitly managed scratchpad, like a cache that you fill yourself.
* **`__syncthreads()`** is a barrier for the threads of the block — like `#pragma omp barrier` for a team.
  It must be reached by all threads of the block: never call it in a branch that only some threads take.
* **Warp shuffles** like `__shfl_down_sync()` exchange registers between the threads of a warp, without memory — the counterpart of SIMD shuffle and permute instructions on the CPU.
* **Atomics** like `atomicAdd()` update global or shared memory without races, as `#pragma omp atomic` does.

There is **no barrier across blocks** within a kernel. Blocks cooperate only through atomics or by ending the kernel: the next kernel in the same stream starts only after the previous one has finished — the end of a kernel is a global barrier, like the end of an OpenMP parallel region.

A **reduction** brings it all together.
It is a tree again, as in the collectives notebook — in four levels, from registers to global memory:

In [ ]:
%%cuda -r -i cuda-check.h -f Icode -o code/cuda-reduce.cu
// the sum over the 32 threads of a warp: a binary tree in registers, without shared memory;
// __shfl_down_sync() reads the variable v of the thread offset lanes further up.
// The mask 0xffffffff says that all 32 lanes take part: the block size must be a multiple of 32.
__device__ double warp_sum(double v) {
    for (int offset = 16; offset > 0; offset /= 2)
        v += __shfl_down_sync(0xffffffff, v, offset);
    return v;                                   // the complete sum is in lane 0
}

__global__ void sum(const double *x, int64_t n, double *result) {
    __shared__ double partial[32];              // one value per warp of the block (at most 32 warps)
    const int lane = threadIdx.x % 32, warp = threadIdx.x / 32;

    // 1. every thread: a private partial sum, with a grid-stride loop
    double v = 0.0;
    for (int64_t i = blockIdx.x * (int64_t)blockDim.x + threadIdx.x; i < n; i += (int64_t)gridDim.x * blockDim.x)
        v += x[i];

    // 2. every warp: tree in registers, lane 0 stores the result of its warp in shared memory
    v = warp_sum(v);
    if (lane == 0)
        partial[warp] = v;
    __syncthreads();                            // barrier of the block: all partial sums are stored

    // 3. the first warp: sum of the partial sums of all warps of the block
    if (warp == 0) {
        v = lane < (int)(blockDim.x / 32) ? partial[lane] : 0.0;
        v = warp_sum(v);
        if (lane == 0)
            atomicAdd(result, v);               // 4. one atomic update per block
    }
}

int main(void) {
    const int64_t n = 1 << 25;
    double *x = (double *)malloc(n * sizeof(double)), *d_x, *d_result;
    for (int64_t i = 0; i < n; ++i)
        x[i] = 1.0 / (i + 1);
    CUDA_CHECK(cudaMalloc(&d_x, n * sizeof(double)));
    CUDA_CHECK(cudaMalloc(&d_result, sizeof(double)));
    CUDA_CHECK(cudaMemcpy(d_x, x, n * sizeof(double), cudaMemcpyHostToDevice));

    double reference = 0.0;                     // the CPU result, summed in order
    for (int64_t i = 0; i < n; ++i)
        reference += x[i];

    const int threads = 256, blocks = 4 * sm_count();
    GpuTimer timer;
    for (int run = 0; run < 3; ++run) {
        double result;
        CUDA_CHECK(cudaMemset(d_result, 0, sizeof(double)));
        timer.start();
        sum<<<blocks, threads>>>(d_x, n, d_result);
        float ms = timer.stop();
        CUDA_CHECK_LAUNCH();
        CUDA_CHECK(cudaMemcpy(&result, d_result, sizeof(double), cudaMemcpyDeviceToHost));
        printf("GPU: %.17f  (CPU - GPU = %9.2e)  %.3f ms, %.0f GB/s\n",
               result, reference - result, ms, n * sizeof(double) / ms / 1e6);
    }
    printf("CPU: %.17f\n", reference);

    CUDA_CHECK(cudaFree(d_x));
    CUDA_CHECK(cudaFree(d_result));
    free(x);
    return 0;
}

> 🤔 Is the GPU result the same in every run? Why does it differ from the CPU result?

The blocks finish in a different order every time, so `atomicAdd()` adds their results in a different order — and floating-point addition is not associative.
A reproducible reduction stores one result per block and adds them in a fixed order, e.g. in a second kernel; libraries like CUB offer this.

## Asynchronous Execution: Streams

Kernel launches and `cudaMemcpyAsync()` only *enqueue* work in a **stream**: a queue whose operations run in order, while operations in different streams may overlap — compute with copies, and copies to the GPU with copies from it, which use different copy engines.
The model is very close to nonblocking MPI:

| Nonblocking MPI | CUDA streams |
|---|---|
| `MPI_Isend()`, `MPI_Irecv()` | `cudaMemcpyAsync(…, stream)`, `kernel<<<blocks, threads, 0, stream>>>(…)` |
| request | stream, event |
| `MPI_Wait()`, `MPI_Test()` | `cudaStreamSynchronize()`, `cudaEventQuery()` |
| overlap of communication and computation | overlap of copies and kernels in different streams |
| registered memory for RDMA | **pinned** host memory (`cudaMallocHost()`): asynchronous copies need it |

The following cell processes the triad in 8 chunks, in 4 streams: while one chunk is being computed, the next is copied to the GPU and the previous one back.

> 🤔 **Predict:** How much faster is the pipelined version? What limits it?

In [ ]:
%%cuda -r -i cuda-check.h -f Icode -o code/cuda-streams.cu
__global__ void triad(double *a, const double *b, const double *c, double s, int64_t n) {
    int64_t i = blockIdx.x * (int64_t)blockDim.x + threadIdx.x;
    if (i < n)
        a[i] = b[i] + s * c[i];
}

int main(void) {
    const int64_t n = 1 << 24;                  // 128 MiB per array
    const size_t bytes = n * sizeof(double);
    const int nchunks = 8, nstreams = 4;
    const int64_t chunk = n / nchunks;

    // pinned (page-locked) host memory: the copy engines of the GPU can access it directly
    double *a, *b, *c, *d_a, *d_b, *d_c;
    CUDA_CHECK(cudaMallocHost(&a, bytes));
    CUDA_CHECK(cudaMallocHost(&b, bytes));
    CUDA_CHECK(cudaMallocHost(&c, bytes));
    for (int64_t i = 0; i < n; ++i) {
        b[i] = 1.0;
        c[i] = 2.0;
    }
    CUDA_CHECK(cudaMalloc(&d_a, bytes));
    CUDA_CHECK(cudaMalloc(&d_b, bytes));
    CUDA_CHECK(cudaMalloc(&d_c, bytes));
    GpuTimer timer;

    // for comparison: a copy from ordinary (pageable) memory goes through a pinned staging buffer
    double *pageable = (double *)malloc(bytes);
    for (int64_t i = 0; i < n; ++i)
        pageable[i] = 1.0;
    timer.start();
    CUDA_CHECK(cudaMemcpy(d_b, pageable, bytes, cudaMemcpyHostToDevice));
    float ms = timer.stop();
    printf("copy to GPU from pageable memory: %5.1f GB/s\n", bytes / ms / 1e6);
    timer.start();
    CUDA_CHECK(cudaMemcpy(d_b, b, bytes, cudaMemcpyHostToDevice));
    ms = timer.stop();
    printf("copy to GPU from pinned memory  : %5.1f GB/s\n\n", bytes / ms / 1e6);
    free(pageable);

    triad<<<(n + 255) / 256, 256>>>(d_a, d_b, d_c, 3.0, n);    // warm-up: the first launch has extra costs
    CUDA_CHECK_LAUNCH();

    // 1. everything in one stream: copy in, compute, copy out - one after the other
    timer.start();
    CUDA_CHECK(cudaMemcpyAsync(d_b, b, bytes, cudaMemcpyHostToDevice, 0));
    CUDA_CHECK(cudaMemcpyAsync(d_c, c, bytes, cudaMemcpyHostToDevice, 0));
    triad<<<(n + 255) / 256, 256>>>(d_a, d_b, d_c, 3.0, n);
    CUDA_CHECK_LAUNCH();
    CUDA_CHECK(cudaMemcpyAsync(a, d_a, bytes, cudaMemcpyDeviceToHost, 0));
    const float ms_serial = timer.stop();

    // 2. pipelined: chunk k goes to stream k % nstreams; work in different streams may overlap
    cudaStream_t streams[nstreams];
    for (int s = 0; s < nstreams; ++s)
        CUDA_CHECK(cudaStreamCreate(&streams[s]));
    timer.start();                              // events in stream 0 wait for all other streams (legacy default stream)
    for (int k = 0; k < nchunks; ++k) {
        cudaStream_t s = streams[k % nstreams];
        const int64_t off = k * chunk;
        CUDA_CHECK(cudaMemcpyAsync(d_b + off, b + off, chunk * sizeof(double), cudaMemcpyHostToDevice, s));
        CUDA_CHECK(cudaMemcpyAsync(d_c + off, c + off, chunk * sizeof(double), cudaMemcpyHostToDevice, s));
        triad<<<(chunk + 255) / 256, 256, 0, s>>>(d_a + off, d_b + off, d_c + off, 3.0, chunk);
        CUDA_CHECK_LAUNCH();
        CUDA_CHECK(cudaMemcpyAsync(a + off, d_a + off, chunk * sizeof(double), cudaMemcpyDeviceToHost, s));
    }
    const float ms_pipelined = timer.stop();

    int64_t wrong = 0;
    for (int64_t i = 0; i < n; ++i)
        wrong += a[i] != 7.0;
    printf("one stream          : %6.2f ms\n", ms_serial);
    printf("%d chunks, %d streams: %6.2f ms  (%lld wrong results)\n", nchunks, nstreams, ms_pipelined, (long long)wrong);

    for (int s = 0; s < nstreams; ++s)
        CUDA_CHECK(cudaStreamDestroy(streams[s]));
    CUDA_CHECK(cudaFree(d_a));
    CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_c));
    CUDA_CHECK(cudaFreeHost(a));
    CUDA_CHECK(cudaFreeHost(b));
    CUDA_CHECK(cudaFreeHost(c));
    return 0;
}

* Pinned memory is copied faster: pageable memory must first be staged through a pinned buffer by the driver.
* The pipeline overlaps the copies to and from the GPU with each other and with the kernels — but the kernel is so short that the copies dominate either way. Overlap hides the shorter operations behind the longest one, it can't make the longest one faster.
  Here the longest one is the copy to the GPU: `b` and `c` go through the same copy engine, one after the other, and that alone is two thirds of the serial copy time. Expect a speedup of at most 1.5.
  As with MPI: overlap helps when there is enough computation to hide the communication behind.

## Outlook: GPUs in MPI Programs

On a GPU cluster, the typical setup is **one MPI process per GPU**: every process selects its GPU by its rank within the node, which a shared-memory communicator provides (see Communicators, Groups and Shared-Memory Windows):

```c
MPI_Comm node;
MPI_Comm_split_type(MPI_COMM_WORLD, MPI_COMM_TYPE_SHARED, 0, MPI_INFO_NULL, &node);
int local_rank, ngpus;
MPI_Comm_rank(node, &local_rank);
cudaGetDeviceCount(&ngpus);
cudaSetDevice(local_rank % ngpus);
```

A halo exchange then consists of a kernel that packs the halo into a buffer, the MPI communication, and a kernel that unpacks it.
**CUDA-aware MPI** libraries accept device pointers directly in `MPI_Send()` and friends and move the data without a detour through host memory, with *GPUDirect RDMA* even directly between the GPU and the network card.
The MPI standard itself says nothing about GPU memory — check your library's documentation (Open MPI: `ompi_info --parsable --all | grep mpi_built_with_cuda_support`).
The datapaths for GPU data and what they require are the topic of [Hybrid MPI+CUDA Programming](hybrid-mpi-cuda.ipynb).

And the many CPU cores of a GPU node? With one process per GPU, they can only be used by threads — the topic of the next notebook, Hybrid MPI+OpenMP Programming.

## Exercises

### Exercise 1: Little's Law and Occupancy

Use the device query, the latency measurement and the Python cells above.

1. How many bytes must be in flight per SM on your GPU to reach its peak bandwidth? Use the latency you measured.
2. A memory-bound kernel loads one `float` per thread and then waits for it. What occupancy does it need at least? And with one `double`, or one `float4` (16 bytes) per thread?
3. The compiler reports 96 registers per thread for this kernel, launched with 256 threads per block. Which occupancy does it reach? What could you do?

<details>
<summary>Solution</summary>

With the B200 numbers (8 TB/s, 148 SMs) and a latency of 500 ns:

1. 8 TB/s × 500 ns = 4 MB in total, ≈ 27 kB per SM.
2. One `float` per thread: 27 kB / 4 B ≈ 6800 threads ≈ 211 warps per SM — more than three times the 64 warps that fit. Even one `double` needs ≈ 106 warps: **full occupancy is not enough**, the kernel needs ILP. With `float4` (16 bytes): ≈ 53 warps, ≈ 83 % occupancy.
   This is why bandwidth-bound kernels load vectors (`float4`, `double2`) or several elements per thread — and the faster the memory, the more parallelism it takes.
3. 96 registers × 32 = 3072 registers per warp; 65 536 / 3072 = 21 warps, i.e. 2 blocks of 8 warps = 16 warps = **25 % occupancy** (`occupancy(256, 96, **limits)`).
   Options: issue more independent loads per thread (ILP), so that 16 warps suffice; reduce the registers with `__launch_bounds__(256, 4)` or `-maxrregcount=64` (at the risk of spills to local memory); restructure the kernel.
</details>

### Exercise 2: π on the GPU

Compute $\pi = \int_0^1 \frac{4}{1+x^2}\,dx$ with the midpoint rule, as in MPI in a Nutshell: implement the kernel with a grid-stride loop and a block reduction as in the reduction example.
Compare the runtime with the MPI version.

<details>
<summary>Solution</summary>

```cpp
__device__ double warp_sum(double v) {
    for (int offset = 16; offset > 0; offset /= 2)
        v += __shfl_down_sync(0xffffffff, v, offset);
    return v;
}

__global__ void pi_kernel(int64_t n, double h, double *sum) {
    __shared__ double partial[32];
    const int lane = threadIdx.x % 32, warp = threadIdx.x / 32;

    double v = 0.0;
    for (int64_t i = blockIdx.x * (int64_t)blockDim.x + threadIdx.x; i < n; i += (int64_t)gridDim.x * blockDim.x) {
        double x = (i + 0.5) * h;
        v += 4.0 / (1.0 + x * x);
    }

    v = warp_sum(v);
    if (lane == 0)
        partial[warp] = v;
    __syncthreads();
    if (warp == 0) {
        v = lane < (int)(blockDim.x / 32) ? partial[lane] : 0.0;
        v = warp_sum(v);
        if (lane == 0)
            atomicAdd(sum, v);
    }
}
```

The kernel is compute-bound — no memory access in the loop — and limited by the FP64 throughput, in particular by the division.
Server GPUs (A100, H100, B200) have half as many FP64 as FP32 lanes, consumer GPUs only 1/64: there, the kernel is much slower in `double`.
</details>

In [ ]:
%%cuda -r -i cuda-check.h -f Icode -o code/cuda-ex-pi.cu
// TODO: every thread sums 4 / (1 + x^2) at the midpoints x = (i + 0.5) * h of its share of the
//       n intervals (grid-stride loop); combine the partial sums of the block as in the
//       reduction example and add the result of the block to *sum with atomicAdd()
__global__ void pi_kernel(int64_t n, double h, double *sum) {
}

int main(void) {
    const int64_t n = 1000000000;
    const double h = 1.0 / n;

    double *d_sum, sum;
    CUDA_CHECK(cudaMalloc(&d_sum, sizeof(double)));
    CUDA_CHECK(cudaMemset(d_sum, 0, sizeof(double)));

    GpuTimer timer;
    timer.start();
    pi_kernel<<<4 * sm_count(), 256>>>(n, h, d_sum);
    float ms = timer.stop();
    CUDA_CHECK_LAUNCH();
    CUDA_CHECK(cudaMemcpy(&sum, d_sum, sizeof(double), cudaMemcpyDeviceToHost));
    printf("pi = %.15f, error %.2e, %.2f ms\n", sum * h, sum * h - 3.14159265358979323846, ms);

    CUDA_CHECK(cudaFree(d_sum));
    return 0;
}

### Exercise 3: A Jacobi Sweep on the GPU

The cell contains the host code for Jacobi sweeps on a grid of 4096 × 4096 inner cells: it runs 10 sweeps on the CPU and on the GPU and compares the results, then times 1000 sweeps on the GPU.

1. Implement the kernel with a 2D grid of 2D blocks: `threadIdx.x` and `blockIdx.x` determine `i`, the contiguous index; `.y` determines `j`.
   The difference to the CPU must be 0.
2. 🤔 **Predict**, then measure: what happens if you swap the roles, i.e. `threadIdx.x` determines `j`?
   (The grid stays the same, since `imax == jmax`.)
3. Compare the bandwidth with the triad. Why does the kernel need only 16 bytes of memory traffic per cell, although it reads five values?

<details>
<summary>Solution</summary>

```cpp
__global__ void jacobi(double *unew, const double *u, int imax, int jmax) {
    const int i = blockIdx.x * blockDim.x + threadIdx.x + 1;     // x: the contiguous index i
    const int j = blockIdx.y * blockDim.y + threadIdx.y + 1;
    if (i <= imax && j <= jmax)
        unew[IDX(i, j)] = 0.25 * (u[IDX(i - 1, j)] + u[IDX(i + 1, j)] + u[IDX(i, j - 1)] + u[IDX(i, j + 1)]);
}
```

2. With `threadIdx.x` along `j`, the 32 threads of a warp access 32 different rows, 32 KiB apart: every load touches 32 sectors instead of 8–9 — uncoalesced, several times slower.
   It is the same mistake as a loop nest with the wrong loop order on the CPU, where the inner loop jumps through memory.
3. The neighbors `u(i ± 1, j)` and `u(i, j ± 1)` were loaded by neighboring threads shortly before, and are found in the L1 and L2 caches; ideally, every value of `u` is read from memory once and every value of `unew` written once: 16 bytes per cell.
   Like the triad, the kernel is memory-bound and should reach a similar bandwidth — which makes the GPU's speedup over a CPU node about the ratio of their memory bandwidths.
   Explicit tiling in shared memory can reduce the cache traffic further, but rarely pays off on current GPUs with large L1 caches.
</details>

In [ ]:
%%cuda -r -i math.h cuda-check.h -f Icode -o code/cuda-ex-jacobi.cu
// cell (i, j) of the grid with imax x jmax inner cells and one layer of boundary cells;
// i is the contiguous index, as in the Laplace solver of the previous notebooks
#define IDX(i, j) ((size_t)(j) * (imax + 2) + (i))

// one Jacobi sweep: thread (threadIdx.x, threadIdx.y) of block (blockIdx.x, blockIdx.y) updates one inner cell
__global__ void jacobi(double *unew, const double *u, int imax, int jmax) {
    // TODO: compute i and j (1 <= i <= imax, 1 <= j <= jmax) from the thread and block indices, then
    //       unew(i, j) = (u(i - 1, j) + u(i + 1, j) + u(i, j - 1) + u(i, j + 1)) / 4
}

int main(void) {
    const int imax = 4096, jmax = 4096;         // inner cells
    const size_t bytes = (size_t)(imax + 2) * (jmax + 2) * sizeof(double);
    const double h = 1.0 / (imax + 1);
    const int check = 10, sweeps = 1000;

    // initial values: u = x + y on the boundary, 0 inside
    double *u = (double *)malloc(bytes), *unew = (double *)malloc(bytes), *result = (double *)malloc(bytes);
    for (int j = 0; j <= jmax + 1; ++j)
        for (int i = 0; i <= imax + 1; ++i)
            u[IDX(i, j)] = unew[IDX(i, j)] = (i == 0 || j == 0 || i == imax + 1 || j == jmax + 1) ? (i + j) * h : 0.0;

    double *d_u, *d_unew;
    CUDA_CHECK(cudaMalloc(&d_u, bytes));
    CUDA_CHECK(cudaMalloc(&d_unew, bytes));
    CUDA_CHECK(cudaMemcpy(d_u, u, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_unew, u, bytes, cudaMemcpyHostToDevice));

    // reference: a few sweeps on the CPU
    for (int s = 0; s < check; ++s) {
        for (int j = 1; j <= jmax; ++j)
            for (int i = 1; i <= imax; ++i)
                unew[IDX(i, j)] = 0.25 * (u[IDX(i - 1, j)] + u[IDX(i + 1, j)] + u[IDX(i, j - 1)] + u[IDX(i, j + 1)]);
        double *tmp = u; u = unew; unew = tmp;
    }

    // the same sweeps on the GPU: 2D blocks of 32 x 8 threads, a 2D grid covering the inner cells
    const dim3 threads(32, 8);
    const dim3 blocks((imax + threads.x - 1) / threads.x, (jmax + threads.y - 1) / threads.y);
    for (int s = 0; s < check; ++s) {
        jacobi<<<blocks, threads>>>(d_unew, d_u, imax, jmax);
        double *tmp = d_u; d_u = d_unew; d_unew = tmp;
    }
    CUDA_CHECK_LAUNCH();
    CUDA_CHECK(cudaMemcpy(result, d_u, bytes, cudaMemcpyDeviceToHost));
    double diff = 0.0;
    for (size_t k = 0; k < bytes / sizeof(double); ++k)
        diff = fmax(diff, fabs(result[k] - u[k]));
    printf("%d sweeps: max. difference to the CPU: %.2e\n", check, diff);

    // performance
    GpuTimer timer;
    timer.start();
    for (int s = 0; s < sweeps; ++s) {
        jacobi<<<blocks, threads>>>(d_unew, d_u, imax, jmax);
        double *tmp = d_u; d_u = d_unew; d_unew = tmp;
    }
    float ms = timer.stop();
    CUDA_CHECK_LAUNCH();
    printf("%d sweeps: %.1f ms, %.0f cell updates per ns, %.0f GB/s (16 bytes per update)\n",
           sweeps, ms, (double)imax * jmax * sweeps / (ms * 1e6), 16.0 * imax * jmax * sweeps / (ms * 1e6));

    CUDA_CHECK(cudaFree(d_u));
    CUDA_CHECK(cudaFree(d_unew));
    free(u);
    free(unew);
    free(result);
    return 0;
}

## Summary

* CPUs minimize the **latency** of one thread; GPUs maximize **throughput** over many threads. Today, a CPU has about as many cores as a GPU has SMs; the GPU has ≈ 3× the FP64 peak, ≈ 14× the memory bandwidth, and almost 800× the threads in flight.
* An **SM** corresponds to a core, a **warp** to a SIMD instruction stream (and a hardware thread), a "CUDA core" to a SIMD lane; a **block** is a team of threads on one SM, the **grid** the whole parallel loop.
* Host and device have **separate memories**; transfers over PCIe are slow compared with device memory — keep the data on the GPU.
* Threads of a warp execute in lockstep: **divergent branches** execute all paths one after the other, and **uncoalesced accesses** waste bandwidth — the GPU versions of masked SIMD code and unused cache line bytes.
* Scheduling: the hardware block scheduler assigns independent blocks to SMs; warp schedulers issue in order, switching between resident warps **every cycle at zero cost**.
* Latency hiding: the CPU finds independent work **within** a thread (out-of-order, ILP, memory-level parallelism), the GPU **across** threads (many resident warps), plus ILP. **Little's law** tells how much is needed.
* **Occupancy** is limited by registers, shared memory and block counts — a trade-off, not a goal in itself.
* Kernel launches and copies are **asynchronous** and ordered in **streams**, like nonblocking MPI operations.

| Call / construct | Purpose |
|---|---|
| `__global__`, `__device__` | kernel (called by the host), device function |
| `kernel<<<blocks, threads, smem, stream>>>(…)` | launch a grid |
| `threadIdx`, `blockIdx`, `blockDim`, `gridDim`, `warpSize` | thread and block indices and sizes |
| `cudaMalloc()`, `cudaFree()`, `cudaMallocHost()`, `cudaMallocManaged()` | device, pinned host and unified memory |
| `cudaMemcpy()`, `cudaMemcpyAsync()`, `cudaMemset()` | copy between host and device |
| `cudaDeviceSynchronize()`, `cudaStreamSynchronize()` | wait for the device / a stream |
| `cudaStreamCreate()`, `cudaEventCreate()`, `cudaEventRecord()`, `cudaEventElapsedTime()` | streams and events, timing |
| `cudaGetLastError()`, `cudaGetErrorString()` | error handling |
| `cudaGetDeviceProperties()`, `cudaDeviceGetAttribute()` | device properties |
| `cudaOccupancyMaxActiveBlocksPerMultiprocessor()`, `cudaFuncGetAttributes()` | occupancy and kernel resources |
| `__shared__`, `__syncthreads()` | shared memory and barrier of a block |
| `__shfl_down_sync()`, `atomicAdd()` | warp shuffle, atomic update |

[Back to the overview](../overview.ipynb)